# Pitwall: fine-tune the router model (free Colab GPU)

Trains a LoRA on Qwen3-0.6B-Base to pick Gearbox's tier (S/M/L) for a prompt, then
exports a quantised GGUF bundle (~400 MB) to download. About 20 minutes.

1. **Runtime -> Change runtime type -> T4 GPU**
2. **Runtime -> Run all**
3. Save the downloaded `<NAME>.zip` into `bundles/` of your Pitwall checkout and unzip it there.

Set the training file in the first code cell: `train_old` (general prompts) or
`train_full` (general + the newer hard prompts, used for retraining after drift).

The notebook only calls the repo's training script (`pitwall.finetune.train`); its
docstring explains each step and why. A failing `!` command does not stop *Run all*, so
every step checks `_exit_code` and stops the notebook on a failure.

In [ ]:
# What to train on. train_old: the general prompts (the first model).
# train_full: general + the newer hard prompts (retraining after drift, M10).
TRAIN = "train_full"
NAME = f"ft-{TRAIN}"  # the bundle's folder and zip name
print(TRAIN, "->", NAME)

In [ ]:
# Stop here if Colab gave this session no GPU (free GPU sessions can end and reconnect
# you to a CPU runtime). Fix: Runtime -> Change runtime type -> T4 GPU, then Run all.
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert _exit_code == 0, "No GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# The Pitwall commit to train from. "main" = latest; a commit id makes the run repeatable.
COMMIT = "main"
!git clone -q https://github.com/Utsav0706/pitwall.git
%cd pitwall
!git checkout -q {COMMIT} && git log -1 --format='training from %h %s'
assert _exit_code == 0, "checkout failed"

In [ ]:
# Colab's own torch is kept (it matches the GPU driver); everything else is pinned.
!pip install -q -e . -r training/requirements-colab.txt
assert _exit_code == 0, "install failed"
# Colab ships torchao 0.10, which peft refuses to load beside (it needs >= 0.16).
# Pitwall doesn't use torchao, so remove it rather than upgrade it against Colab's torch.
!pip uninstall -q -y torchao

In [ ]:
# Rebuild the training data exactly: fetch Gearbox's files from the pinned commit, build
# the pairs, and stop if anything differs from the committed dvc.lock.
!dvc update -q -R data/gearbox
assert _exit_code == 0, "dvc update failed"
!dvc repro -q
assert _exit_code == 0, "dvc repro failed"
!git diff --exit-code --stat
assert _exit_code == 0, "data differs from dvc.lock - stop"
!pitwall data show

In [ ]:
!python -m pitwall.finetune.train     --train data/derived/{TRAIN}.jsonl     --out bundles/{NAME}     --llama-cpp .llama.cpp
assert _exit_code == 0, "training failed - see the output above"

In [ ]:
!pitwall bundle check bundles/{NAME}
assert _exit_code == 0, "bundle check failed"
import shutil

archive = shutil.make_archive(NAME, "zip", "bundles", NAME)
print(archive)

In [ ]:
# Download. If the browser download stalls (~400 MB), use the Drive cell below instead.
from google.colab import files

files.download(f"{NAME}.zip")

In [ ]:
# Optional: copy to Google Drive instead (asks for Drive access).
# from google.colab import drive
# drive.mount("/content/drive")
# !cp {NAME}.zip /content/drive/MyDrive/